# Phase 6 — ML Preprocessing Pipeline

## H&M Personalized Fashion Recommendations → Customer Purchase Prediction

### Objective

In Phase 5, we converted the H&M transaction history into a **time-aware supervised learning problem**:

> **Predict whether a customer will make at least one purchase during the next 30 days.**

In this phase, we prepare the Phase 5 datasets for machine learning.

### What we will do

1. Load the temporal train/validation/test datasets.
2. Separate features, target, and customer identifiers.
3. Identify numerical and categorical features.
4. Audit missing values and feature types.
5. Build a leakage-safe preprocessing pipeline.
6. Handle missing numerical values using median imputation.
7. Handle missing categorical values using an explicit `"Unknown"` category.
8. One-hot encode categorical variables.
9. Standardize numerical variables.
10. Build a robust-scaling alternative for outlier-heavy features.
11. Compare the dimensionality before and after encoding.
12. Extract the generated feature names.
13. Apply **PCA to the continuous numerical feature space**.
14. Analyze explained variance and cumulative explained variance.
15. Verify that no missing values remain after preprocessing.
16. Save fitted preprocessing pipelines and feature metadata.

> **Important:** All preprocessing is fitted **only on the training data**. Validation and test data are transformed using the already-fitted training transformations.

### Why this phase matters

A machine-learning model should never learn preprocessing parameters from the validation or test sets. Doing so creates **data leakage** and produces overly optimistic evaluation results.

The pipeline developed here will be reused in Phase 7 for baseline model training.


## 6.1 Overall Preprocessing Architecture

```text
                    Phase 5 Data
                         │
              ┌──────────┴──────────┐
              │                     │
        Numerical Features    Categorical Features
              │                     │
       Median Imputation      Constant Imputation
              │                     │
       StandardScaler          OneHotEncoder
              │                     │
              └──────────┬──────────┘
                         │
                  Processed Matrix
                         │
              ┌──────────┴──────────┐
              │                     │
          No PCA                 PCA Path
              │                     │
       Full feature space    PCA on numerical
                            continuous features
```

### Important PCA design decision

The complete encoded feature matrix contains one-hot encoded categorical variables and is therefore naturally **sparse**.

Standard `sklearn.decomposition.PCA` expects dense input and can become very memory-intensive when applied to a large one-hot encoded matrix.

Therefore:

- **Full ML preprocessing:** numerical + categorical features are retained.
- **PCA experiment:** PCA is applied to the **continuous numerical feature space only**.
- Categorical variables remain one-hot encoded.
- If dimensionality reduction of the sparse categorical space is required later, **TruncatedSVD** is more appropriate than PCA. That is a separate experiment and is intentionally not treated as PCA here.


## 6.2 Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import os
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import explained_variance_score

RANDOM_STATE = 42

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_phase5.parquet"
VAL_PATH = PROCESSED_DIR / "validation_phase5.parquet"
TEST_PATH = PROCESSED_DIR / "test_phase5.parquet"

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data:", PROCESSED_DIR.resolve())
print("Models directory:", MODELS_DIR.resolve())


## 6.3 Load Phase 5 Datasets

Phase 5 produced three temporally separated datasets:

- `train_phase5.parquet`
- `validation_phase5.parquet`
- `test_phase5.parquet`

These files contain the engineered customer-level features and the target.

We do **not** rebuild the target here. Target engineering was completed in Phase 5.


In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())


## 6.4 Inspect Dataset Schema

Before creating a preprocessing pipeline, we verify:

- column names
- data types
- target presence
- identifier presence
- consistency between train/validation/test

This is an important engineering checkpoint because preprocessing should receive the same feature schema at training and inference time.


In [ ]:
print("Train columns:")
for col in train_df.columns:
    print(f"- {col}: {train_df[col].dtype}")

print("\nColumn consistency checks:")

train_cols = set(train_df.columns)
val_cols = set(val_df.columns)
test_cols = set(test_df.columns)

print("Train == Validation:", train_cols == val_cols)
print("Train == Test:", train_cols == test_cols)

assert train_cols == val_cols == test_cols, "Feature schema mismatch detected."


## 6.5 Separate Identifier, Features, and Target

### Target

The target is:

- `target = 1` → customer purchases at least once in the following 30-day prediction window.
- `target = 0` → customer does not purchase during that window.

### Identifier

`customer_id` is an identifier, not a predictive feature.

We remove it from the ML feature matrix because:

- it has extremely high cardinality,
- it does not represent customer behavior,
- one-hot encoding it would create an enormous feature space,
- it can encourage memorization rather than generalization.

The ID is retained separately so predictions can later be mapped back to customers.


In [ ]:
TARGET_COL = "target"
ID_COL = "customer_id"

assert TARGET_COL in train_df.columns, "Target column not found."
assert ID_COL in train_df.columns, "Customer ID column not found."

customer_id_train = train_df[ID_COL].copy()
customer_id_val = val_df[ID_COL].copy()
customer_id_test = test_df[ID_COL].copy()

X_train = train_df.drop(columns=[TARGET_COL, ID_COL]).copy()
X_val = val_df.drop(columns=[TARGET_COL, ID_COL]).copy()
X_test = test_df.drop(columns=[TARGET_COL, ID_COL]).copy()

y_train = train_df[TARGET_COL].astype("int8").copy()
y_val = val_df[TARGET_COL].astype("int8").copy()
y_test = test_df[TARGET_COL].astype("int8").copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


## 6.6 Verify Feature Alignment

The model must see the same raw feature columns in the same logical schema across all splits.

The `ColumnTransformer` will later guarantee consistent transformed feature ordering.


In [ ]:
assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("Feature alignment verified.")
print("Number of raw ML features:", X_train.shape[1])


## 6.7 Identify Numerical and Categorical Features

We automatically identify feature types from the training data.

### Numerical features

Examples include:

- recency
- total spend
- purchase frequency
- number of unique articles
- average price
- recent spending
- customer tenure
- diversity measures
- age
- log-transformed numerical variables
- outlier indicator variables

### Categorical features

Examples include:

- club membership status
- fashion news frequency
- preferred product group
- active status
- FN-related categorical fields

The training set determines the feature types. We do not infer them independently from validation/test data.


In [ ]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


## 6.8 Missing-Value Audit

Missing values are expected in customer-level retail data.

For example:

- customer demographic information may be unavailable,
- membership fields may contain missing values,
- derived behavioral features can be undefined for customers with insufficient history.

### Strategy

**Numerical**
- Use median imputation.
- Median is less sensitive to extreme values than mean.

**Categorical**
- Replace missing values with `"Unknown"`.
- Then one-hot encode.

### Leakage rule

Imputation statistics are learned from the **training set only**.


In [ ]:
missing_train = X_train.isna().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

missing_val = X_val.isna().sum()
missing_val = missing_val[missing_val > 0].sort_values(ascending=False)

missing_test = X_test.isna().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)

print("Missing values in training set:")
display(missing_train.to_frame("missing_count"))

print("Missing values in validation set:")
display(missing_val.to_frame("missing_count"))

print("Missing values in test set:")
display(missing_test.to_frame("missing_count"))


## 6.9 Check Cardinality of Categorical Variables

One-hot encoding works well for moderate-cardinality categorical features.

However, high-cardinality variables can create a very large sparse matrix.

We inspect the number of unique values before encoding.


In [ ]:
if categorical_features:
    cardinality = pd.DataFrame({
        "feature": categorical_features,
        "unique_values": [
            X_train[col].nunique(dropna=False)
            for col in categorical_features
        ]
    }).sort_values("unique_values", ascending=False)

    display(cardinality)
else:
    print("No categorical features detected.")


## 6.10 One-Hot Encoder Compatibility

Different versions of scikit-learn use different parameter names for sparse output.

We use a small compatibility block so the notebook works with both newer and older versions.


In [ ]:
try:
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

print("OneHotEncoder configured successfully.")


## 6.11 Standard Numerical Preprocessing

For the standard pipeline:

```text
Numerical
   ↓
Median Imputation
   ↓
StandardScaler
```

Standardization transforms a numerical variable approximately as:

\[
z = \frac{x - \mu}{\sigma}
\]

where:

- \(x\) = original value
- \(\mu\) = training-set mean
- \(\sigma\) = training-set standard deviation

This is particularly useful for:

- Logistic Regression
- SVM
- KNN
- SGD-based models
- PCA
- neural networks

Tree-based models generally do not require scaling, but keeping a standardized pipeline makes the feature representation reusable across several model families.


In [ ]:
numeric_pipeline_standard = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("onehot", ohe)
    ]
)

preprocessor_standard = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline_standard, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print(preprocessor_standard)


## 6.12 Fit Preprocessing ONLY on Training Data

This is one of the most important steps in the project.

### Correct

```text
X_train → fit_transform()
X_val   → transform()
X_test  → transform()
```

### Incorrect

```text
X_train + X_val + X_test → fit_transform()
```

The incorrect approach allows information from validation/test distributions to influence:

- imputation values,
- scaling parameters,
- category vocabulary.

That is a form of data leakage.


In [ ]:
X_train_processed = preprocessor_standard.fit_transform(X_train)

X_val_processed = preprocessor_standard.transform(X_val)
X_test_processed = preprocessor_standard.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nSparse matrix:", sp.issparse(X_train_processed))


## 6.13 Inspect Sparse Matrix Properties

One-hot encoding usually creates many zero values.

For example, if a categorical variable has 20 possible categories, each row activates only a small number of those 20 columns.

Storing the entire matrix as a dense NumPy array would waste memory.

Therefore, we preserve the sparse representation whenever possible.


In [ ]:
def matrix_memory_report(matrix, name):
    if sp.issparse(matrix):
        memory_mb = (
            matrix.data.nbytes
            + matrix.indices.nbytes
            + matrix.indptr.nbytes
        ) / (1024 ** 2)

        print(f"{name}:")
        print("  Type:", type(matrix).__name__)
        print("  Shape:", matrix.shape)
        print(f"  Non-zero values: {matrix.nnz:,}")
        print(f"  Sparsity: {100 * (1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])):.2f}%")
        print(f"  Approx. sparse memory: {memory_mb:.2f} MB")
    else:
        memory_mb = matrix.nbytes / (1024 ** 2)

        print(f"{name}:")
        print("  Type:", type(matrix).__name__)
        print("  Shape:", matrix.shape)
        print(f"  Approx. dense memory: {memory_mb:.2f} MB")

matrix_memory_report(X_train_processed, "Processed training matrix")
matrix_memory_report(X_val_processed, "Processed validation matrix")
matrix_memory_report(X_test_processed, "Processed test matrix")


## 6.14 Extract Transformed Feature Names

After one-hot encoding, the number of columns increases.

For example:

```text
club_member_status
```

could become:

```text
club_member_status_ACTIVE
club_member_status_PRE-CREATE
club_member_status_UNKNOWN
```

`ColumnTransformer.get_feature_names_out()` gives us the final feature names in exactly the order used by the transformed matrix.


In [ ]:
feature_names_standard = preprocessor_standard.get_feature_names_out()

print("Number of transformed features:", len(feature_names_standard))

feature_name_df = pd.DataFrame({
    "feature_index": np.arange(len(feature_names_standard)),
    "feature_name": feature_names_standard
})

display(feature_name_df.head(30))


## 6.15 Verify No Missing Values Remain

After fitting the preprocessing pipeline, no missing values should remain in the transformed feature matrix.

This is an important validation before passing the matrix to a machine-learning algorithm.


In [ ]:
def check_missing_after_transform(matrix, name):
    if sp.issparse(matrix):
        missing = np.isnan(matrix.data).sum()
        print(f"{name}: NaN values in stored data = {missing}")
        return missing == 0
    else:
        missing = np.isnan(matrix).sum()
        print(f"{name}: NaN values = {missing}")
        return missing == 0

assert check_missing_after_transform(
    X_train_processed, "Train"
)

assert check_missing_after_transform(
    X_val_processed, "Validation"
)

assert check_missing_after_transform(
    X_test_processed, "Test"
)

print("No NaN values remain after preprocessing.")


## 6.16 Standard Scaling vs Robust Scaling

Phase 4 identified heavy-tailed variables and created outlier indicators.

We should **not delete customers simply because their behavior is unusual**. A high-spending customer may be a perfectly legitimate and important customer.

Instead, we can compare two scaling strategies:

### StandardScaler

Uses mean and standard deviation.

Advantages:
- common default,
- useful for many linear models,
- appropriate when distributions are reasonably well behaved.

### RobustScaler

Uses median and interquartile range.

Advantages:
- less sensitive to extreme values,
- useful when numerical variables contain substantial outliers.

We will preserve both options and let later model experiments determine which performs better.


In [ ]:
try:
    ohe_robust = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    ohe_robust = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

numeric_pipeline_robust = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", RobustScaler())
    ]
)

categorical_pipeline_robust = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("onehot", ohe_robust)
    ]
)

preprocessor_robust = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline_robust, numeric_features),
        ("cat", categorical_pipeline_robust, categorical_features)
    ],
    remainder="drop"
)

X_train_robust = preprocessor_robust.fit_transform(X_train)
X_val_robust = preprocessor_robust.transform(X_val)
X_test_robust = preprocessor_robust.transform(X_test)

print("Robust train shape:", X_train_robust.shape)
print("Robust validation shape:", X_val_robust.shape)
print("Robust test shape:", X_test_robust.shape)


## 6.17 Important PCA Consideration

### Why not directly run PCA on the complete encoded matrix?

The standard preprocessing matrix can contain:

- continuous numerical features,
- many one-hot categorical features,
- potentially millions of customer rows.

One-hot encoded data is sparse.

Standard PCA is based on operations that generally require a dense matrix. Converting a very large sparse matrix to dense form can cause extreme memory consumption.

Therefore, this project uses PCA on the **continuous numerical features**.

This still gives us a meaningful PCA experiment because the behavioral variables can be correlated:

- total spend ↔ recent spend
- total items ↔ recent items
- purchase days ↔ purchase rate
- unique articles ↔ diversity
- tenure ↔ active months

PCA can identify lower-dimensional combinations of these correlated numerical variables.

### PCA workflow

```text
Numerical features
       ↓
Median imputation
       ↓
Standardization
       ↓
PCA
       ↓
Principal components
```


## 6.18 Prepare Numerical Data for PCA

We first create a numerical-only preprocessing pipeline.

The PCA model must also be fitted only on the training data.


In [ ]:
numeric_imputer_scaler = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

X_train_num_scaled = numeric_imputer_scaler.fit_transform(
    X_train[numeric_features]
)

X_val_num_scaled = numeric_imputer_scaler.transform(
    X_val[numeric_features]
)

X_test_num_scaled = numeric_imputer_scaler.transform(
    X_test[numeric_features]
)

print("Scaled numerical train shape:", X_train_num_scaled.shape)
print("Scaled numerical validation shape:", X_val_num_scaled.shape)
print("Scaled numerical test shape:", X_test_num_scaled.shape)


## 6.19 Fit PCA and Analyze Explained Variance

We initially fit PCA with all available numerical components.

The purpose is to determine how many principal components are required to explain a large proportion of the numerical feature variance.

A common engineering choice is to investigate the number of components required for:

- 80% explained variance
- 90% explained variance
- 95% explained variance
- 99% explained variance

We should not blindly choose a number such as 2 or 3 without examining the variance structure of the actual dataset.


In [ ]:
n_numeric = X_train_num_scaled.shape[1]

pca_full = PCA(
    n_components=n_numeric,
    svd_solver="full",
    random_state=RANDOM_STATE
)

X_train_pca_full = pca_full.fit_transform(X_train_num_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

pca_summary = pd.DataFrame({
    "component": np.arange(1, n_numeric + 1),
    "explained_variance_ratio": explained_variance,
    "cumulative_explained_variance": cumulative_variance
})

display(pca_summary)


## 6.20 PCA Explained-Variance Plot

The cumulative explained variance curve helps us decide how many components provide an acceptable representation of the original numerical feature space.


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    np.arange(1, n_numeric + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(
    0.80,
    linestyle="--",
    label="80% variance"
)

plt.axhline(
    0.90,
    linestyle="--",
    label="90% variance"
)

plt.axhline(
    0.95,
    linestyle="--",
    label="95% variance"
)

plt.axhline(
    0.99,
    linestyle="--",
    label="99% variance"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA — Cumulative Explained Variance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 6.21 Automatically Identify Component Counts

We identify the smallest number of components needed to reach selected explained-variance thresholds.

This makes the PCA decision data-driven.


In [ ]:
def components_for_variance(cumulative_variance, threshold):
    return int(np.argmax(cumulative_variance >= threshold) + 1)

variance_targets = [0.80, 0.90, 0.95, 0.99]

pca_component_report = pd.DataFrame({
    "target_explained_variance": variance_targets,
    "required_components": [
        components_for_variance(cumulative_variance, threshold)
        for threshold in variance_targets
    ]
})

display(pca_component_report)


## 6.22 Build a 95% Variance PCA Pipeline

For experimentation, we use the smallest number of principal components that explains at least 95% of the variance in the training numerical feature space.

The exact number is determined from the previous cell rather than hard-coded.

> **Important:** PCA components are learned from the training set only.


In [ ]:
PCA_VARIANCE_TARGET = 0.95

n_components_95 = components_for_variance(
    cumulative_variance,
    PCA_VARIANCE_TARGET
)

print(
    f"Components needed for {PCA_VARIANCE_TARGET:.0%} variance:",
    n_components_95
)

pca_95 = PCA(
    n_components=n_components_95,
    svd_solver="full",
    random_state=RANDOM_STATE
)

X_train_num_pca = pca_95.fit_transform(X_train_num_scaled)
X_val_num_pca = pca_95.transform(X_val_num_scaled)
X_test_num_pca = pca_95.transform(X_test_num_scaled)

print("Original numerical dimensions:", n_numeric)
print("PCA numerical dimensions:", X_train_num_pca.shape[1])


## 6.23 Compare Numerical Dimensions Before and After PCA

PCA does not necessarily reduce the complete ML feature space because categorical one-hot features are intentionally kept separate.

The comparison here is specifically for the **continuous numerical block**.


In [ ]:
pca_dimension_report = pd.DataFrame({
    "representation": [
        "Original numerical features",
        "PCA numerical features"
    ],
    "dimensions": [
        X_train_num_scaled.shape[1],
        X_train_num_pca.shape[1]
    ]
})

display(pca_dimension_report)

reduction_pct = (
    1 -
    X_train_num_pca.shape[1] / X_train_num_scaled.shape[1]
) * 100

print(f"Numerical dimensionality reduction: {reduction_pct:.2f}%")
print(
    f"Explained variance retained: "
    f"{pca_95.explained_variance_ratio_.sum():.4f}"
)


## 6.24 Inspect PCA Component Loadings

PCA components are linear combinations of the original numerical variables.

The loading magnitude indicates how strongly an original variable contributes to a component.

This is useful for interpretation and for understanding what behavioral dimensions the model is capturing.


In [ ]:
pca_feature_names = np.array(numeric_features)

loadings = pd.DataFrame(
    pca_95.components_.T,
    index=pca_feature_names,
    columns=[
        f"PC{i+1}"
        for i in range(n_components_95)
    ]
)

display(loadings.head(20))


## 6.25 Top Contributing Features per Principal Component

For each principal component, we display the variables with the largest absolute loadings.

This provides an interpretable view of the behavioral dimensions represented by PCA.


In [ ]:
TOP_N_LOADINGS = 10

for component in loadings.columns:
    top_features = (
        loadings[component]
        .abs()
        .sort_values(ascending=False)
        .head(TOP_N_LOADINGS)
        .index
    )

    print(f"\n{component}")
    display(
        loadings.loc[top_features, [component]]
        .sort_values(component, key=np.abs, ascending=False)
    )


## 6.26 Build a Combined PCA + Categorical Representation

We can create a PCA-based representation in which:

- numerical features are imputed,
- numerical features are standardized,
- numerical features are reduced using PCA,
- categorical features are imputed,
- categorical features are one-hot encoded.

This is an **experimental representation**. It will be compared with the standard non-PCA representation in later model experiments.

Because the PCA numerical output is dense while one-hot categorical output is sparse, the final `ColumnTransformer` may choose a sparse representation depending on the resulting density.


In [ ]:
numeric_pca_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components_95,
            svd_solver="full",
            random_state=RANDOM_STATE
        ))
    ]
)

try:
    ohe_pca = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    ohe_pca = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

categorical_pca_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("onehot", ohe_pca)
    ]
)

preprocessor_pca = ColumnTransformer(
    transformers=[
        ("num_pca", numeric_pca_pipeline, numeric_features),
        ("cat", categorical_pca_pipeline, categorical_features)
    ],
    remainder="drop"
)

X_train_pca_processed = preprocessor_pca.fit_transform(X_train)
X_val_pca_processed = preprocessor_pca.transform(X_val)
X_test_pca_processed = preprocessor_pca.transform(X_test)

print("PCA representation:")
print("Train:", X_train_pca_processed.shape)
print("Validation:", X_val_pca_processed.shape)
print("Test:", X_test_pca_processed.shape)
print("Sparse:", sp.issparse(X_train_pca_processed))


## 6.27 Compare Standard vs PCA Feature Representations

We now compare:

1. **Standard representation**
   - numerical imputation + scaling
   - categorical imputation + one-hot encoding

2. **PCA representation**
   - numerical imputation + scaling + PCA
   - categorical imputation + one-hot encoding

This comparison will be useful in Phase 7 and later model experiments.


In [ ]:
representation_comparison = pd.DataFrame({
    "representation": [
        "Standard preprocessing",
        "PCA preprocessing"
    ],
    "train_rows": [
        X_train_processed.shape[0],
        X_train_pca_processed.shape[0]
    ],
    "transformed_features": [
        X_train_processed.shape[1],
        X_train_pca_processed.shape[1]
    ],
    "sparse": [
        sp.issparse(X_train_processed),
        sp.issparse(X_train_pca_processed)
    ]
})

display(representation_comparison)


## 6.28 Verify Unknown Categories Are Handled Safely

A customer category can appear in validation/test even if it was not present in training.

Because we use:

```python
OneHotEncoder(handle_unknown="ignore")
```

the pipeline will not fail when an unseen category appears.

Instead, the corresponding one-hot block will contain zeros.

This is important for real-world inference, where future data can contain categories not observed during training.


In [ ]:
print("Validation transformed successfully.")
print("Test transformed successfully.")
print("Unknown-category handling is enabled through handle_unknown='ignore'.")


## 6.29 Optional: Save Transformed Matrices

### Storage warning

The H&M dataset is large.

The transformed feature matrices can consume substantial disk space even when stored sparsely.

Therefore, this notebook does **not** automatically write the matrices unless explicitly enabled.

The recommended approach is to save the fitted preprocessing pipeline and regenerate transformations when required.

Set `SAVE_TRANSFORMED_MATRICES = True` only when you have sufficient disk space.


In [ ]:
SAVE_TRANSFORMED_MATRICES = False

TRANSFORMED_DIR = PROCESSED_DIR / "phase6_transformed"
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_TRANSFORMED_MATRICES:
    if sp.issparse(X_train_processed):
        sp.save_npz(
            TRANSFORMED_DIR / "X_train_standard_phase6.npz",
            X_train_processed
        )
        sp.save_npz(
            TRANSFORMED_DIR / "X_val_standard_phase6.npz",
            X_val_processed
        )
        sp.save_npz(
            TRANSFORMED_DIR / "X_test_standard_phase6.npz",
            X_test_processed
        )
    else:
        np.save(
            TRANSFORMED_DIR / "X_train_standard_phase6.npy",
            X_train_processed
        )
        np.save(
            TRANSFORMED_DIR / "X_val_standard_phase6.npy",
            X_val_processed
        )
        np.save(
            TRANSFORMED_DIR / "X_test_standard_phase6.npy",
            X_test_processed
        )

    print("Standard transformed matrices saved.")
else:
    print(
        "Transformed matrices were not saved. "
        "Set SAVE_TRANSFORMED_MATRICES=True if required."
    )


## 6.30 Save Feature Names

Feature names are saved separately so that later we can:

- inspect model coefficients,
- perform feature importance analysis,
- debug predictions,
- explain one-hot encoded variables,
- map transformed columns back to their source features.


In [ ]:
FEATURE_NAMES_PATH = PROCESSED_DIR / "feature_names_standard_phase6.csv"
PCA_FEATURE_NAMES_PATH = PROCESSED_DIR / "feature_names_pca_phase6.csv"

pd.DataFrame({
    "feature_index": np.arange(len(feature_names_standard)),
    "feature_name": feature_names_standard
}).to_csv(FEATURE_NAMES_PATH, index=False)

pca_feature_names_full = preprocessor_pca.get_feature_names_out()

pd.DataFrame({
    "feature_index": np.arange(len(pca_feature_names_full)),
    "feature_name": pca_feature_names_full
}).to_csv(PCA_FEATURE_NAMES_PATH, index=False)

print("Saved:")
print(FEATURE_NAMES_PATH)
print(PCA_FEATURE_NAMES_PATH)


## 6.31 Save Fitted Preprocessing Pipelines

The fitted preprocessing objects contain learned parameters such as:

- numerical medians,
- scaling statistics,
- categorical vocabularies,
- PCA components.

Saving them allows the exact same transformation to be applied during:

- validation,
- testing,
- future inference,
- deployment.

Never fit a new scaler or encoder independently during inference.


In [ ]:
STANDARD_PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_standard_phase6.joblib"
ROBUST_PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_robust_phase6.joblib"
PCA_PREPROCESSOR_PATH = MODELS_DIR / "preprocessor_pca_phase6.joblib"
NUMERIC_PCA_PATH = MODELS_DIR / "numeric_pca_phase6.joblib"

joblib.dump(preprocessor_standard, STANDARD_PREPROCESSOR_PATH)
joblib.dump(preprocessor_robust, ROBUST_PREPROCESSOR_PATH)
joblib.dump(preprocessor_pca, PCA_PREPROCESSOR_PATH)
joblib.dump(
    {
        "imputer_scaler": numeric_imputer_scaler,
        "pca": pca_95,
        "numeric_features": numeric_features,
        "n_components": n_components_95
    },
    NUMERIC_PCA_PATH
)

print("Saved preprocessing artifacts:")
print("-", STANDARD_PREPROCESSOR_PATH)
print("-", ROBUST_PREPROCESSOR_PATH)
print("-", PCA_PREPROCESSOR_PATH)
print("-", NUMERIC_PCA_PATH)


## 6.32 Final Leakage and Integrity Checks

Before finishing Phase 6, verify:

### Check 1 — Customer ID is not a model feature

`customer_id` must not appear in `X_train`.

### Check 2 — Target is not a model feature

`target` must not appear in `X_train`.

### Check 3 — Validation/test are only transformed

Their data was never used to fit:

- imputers,
- scalers,
- one-hot vocabularies,
- PCA.

### Check 4 — Feature counts match

Train, validation, and test must have identical transformed dimensions.

### Check 5 — No NaNs

The final transformed matrices should contain no NaN values.


In [ ]:
assert ID_COL not in X_train.columns
assert TARGET_COL not in X_train.columns

assert X_train_processed.shape[1] == X_val_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert X_train_pca_processed.shape[1] == X_val_pca_processed.shape[1]
assert X_train_pca_processed.shape[1] == X_test_pca_processed.shape[1]

assert check_missing_after_transform(
    X_train_processed, "Final train"
)

assert check_missing_after_transform(
    X_val_processed, "Final validation"
)

assert check_missing_after_transform(
    X_test_processed, "Final test"
)

print("\nAll Phase 6 integrity checks passed.")


# Phase 6 — Final Summary

## What we accomplished

### 1. Data separation
We separated:

- `customer_id`
- target
- ML features

### 2. Feature typing
We identified:

- numerical features
- categorical features

### 3. Missing-value handling
We implemented:

- median imputation for numerical features,
- `"Unknown"` imputation for categorical features.

### 4. Categorical encoding
We used one-hot encoding with:

```text
handle_unknown="ignore"
```

This makes the pipeline robust to unseen future categories.

### 5. Numerical scaling
We created:

- StandardScaler pipeline
- RobustScaler pipeline

### 6. PCA
We applied PCA to the continuous numerical feature space and selected the number of components required to retain approximately 95% of training variance.

### 7. Leakage prevention
All preprocessing parameters were learned from **training data only**.

### 8. Artifact persistence
We saved:

- fitted preprocessing pipelines,
- PCA pipeline,
- transformed feature names.

---

# Phase 6 Deliverables

```text
models/
├── preprocessor_standard_phase6.joblib
├── preprocessor_robust_phase6.joblib
├── preprocessor_pca_phase6.joblib
└── numeric_pca_phase6.joblib

data/processed/
├── feature_names_standard_phase6.csv
└── feature_names_pca_phase6.csv
```

Optional transformed matrices can also be saved if required.

---

# What comes next — Phase 7

## Baseline Model + Bias–Variance Analysis

In Phase 7, we will finally start training machine-learning models.

The baseline stage should include:

1. Establish a simple baseline.
2. Train Logistic Regression.
3. Establish a Dummy Classifier reference.
4. Evaluate:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - ROC-AUC
   - PR-AUC
   - Confusion Matrix
5. Compare training vs validation performance.
6. Diagnose:
   - underfitting,
   - overfitting,
   - bias,
   - variance.
7. Establish the first meaningful benchmark that all later models must beat.

> **Phase 6 is preprocessing only. Model selection and final performance claims should begin in Phase 7.**
